In [1]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objs as go
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, StackingClassifier
from sklearn.metrics import f1_score, matthews_corrcoef, auc, precision_recall_curve, accuracy_score, confusion_matrix, classification_report, average_precision_score
from keras.models import Model
from keras.layers import Input, Dense, Conv1D, MaxPooling1D, GRU, LSTM, Bidirectional, Dropout, Flatten, Attention, BatchNormalization
from keras.optimizers import Adam
from keras.utils import to_categorical
from sklearn.linear_model import LogisticRegression
from imblearn.over_sampling import SMOTE
from sklearn.utils.class_weight import compute_class_weight

In [2]:
# Load the dataset
data = pd.read_parquet('train_data.parquet')
data.head(2000)

,id1,id2,id3,id4,id5,y,f1,f2,f3,f4,...,f357,f358,f359,f360,f361,f362,f363,f364,f365,f366
0,1366776_189706075_16-23_2023-11-02 22:22:00.042,1366776,189706075,2023-11-02 22:22:00.042,2023-11-02,0,1.0,None,None,None,...,None,-9999.0,0.0,None,28.0,0.0,0.0,337.0,0.0,0.0
1,1366776_89227_16-23_2023-11-01 23:51:24.999,1366776,89227,2023-11-01 23:51:24.999,2023-11-01,0,1.0,None,None,None,...,None,None,0.0,None,87.0,0.0,0.0,1010.0,2.0,0.0019801980198019
2,1366776_35046_16-23_2023-11-01 00:30:59.797,1366776,35046,2023-11-01 00:30:59.797,2023-11-01,0,1.0,None,None,None,...,None,None,0.0,None,23.0,0.0,0.0,1010.0,2.0,0.0019801980198019
3,1366776_6275451_16-23_2023-11-02 22:21:32.261,1366776,6275451,2023-11-02 22:21:32.261,2023-11-02,0,1.0,None,None,None,...,None,-9999.0,0.0,None,277.0,1.0,0.003610108303249,337.0,0.0,0.0
4,1366776_78053_16-23_2023-11-02 22:21:34.799,1366776,78053,2023-11-02 22:21:34.799,2023-11-02,0,1.0,None,None,None,...,None,-9999.0,0.0,None,359.0,0.0,0.0,337.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1995,1682204_64062_16-23_2023-11-03 15:30:29.977,1682204,64062,2023-11-03 15:30:29.977,2023-11-03,0,40.0,None,None,None,...,-0.0182848079755296,0.0426645519429024,0.0,None,27.0,17.0,0.6296296296296297,458.0,190.0,0.4148471615720524
1996,1682204_189706075_16-23_2023-11-03 15:23:30.00...,1682204,189706075,2023-11-03 15:23:30.000078,2023-11-03,1,40.0,None,None,None,...,-0.0145408605613218,0.0610716143575515,0.0,None,24.0,7.0,0.2916666666666667,55.0,39.0,0.7090909090909091
1997,1682204_9914_16-23_2023-11-03 15:23:56.000027,1682204,9914,2023-11-03 15:23:56.000027,2023-11-03,1,40.0,None,None,None,...,0.0100555607082228,0.0464871370543102,0.0,None,138.0,52.0,0.3768115942028985,231.0,96.0,0.4155844155844156
1998,1682204_84457_16-23_2023-11-03 15:26:21.000473,1682204,84457,2023-11-03 15:26:21.000473,2023-11-03,1,40.0,None,None,None,...,-0.0187831003184442,0.0438272340763698,0.0,None,None,None,None,59.0,25.0,0.423728813559322


In [3]:
# 1. Drop duplicates
numeric_data=data

In [4]:
numeric_data.head()

,id1,id2,id3,id4,id5,y,f1,f2,f3,f4,...,f357,f358,f359,f360,f361,f362,f363,f364,f365,f366
0,1366776_189706075_16-23_2023-11-02 22:22:00.042,1366776,189706075,2023-11-02 22:22:00.042,2023-11-02,0,1.0,None,None,None,...,None,-9999.0,0.0,None,28.0,0.0,0.0,337.0,0.0,0.0
1,1366776_89227_16-23_2023-11-01 23:51:24.999,1366776,89227,2023-11-01 23:51:24.999,2023-11-01,0,1.0,None,None,None,...,None,None,0.0,None,87.0,0.0,0.0,1010.0,2.0,0.0019801980198019
2,1366776_35046_16-23_2023-11-01 00:30:59.797,1366776,35046,2023-11-01 00:30:59.797,2023-11-01,0,1.0,None,None,None,...,None,None,0.0,None,23.0,0.0,0.0,1010.0,2.0,0.0019801980198019
3,1366776_6275451_16-23_2023-11-02 22:21:32.261,1366776,6275451,2023-11-02 22:21:32.261,2023-11-02,0,1.0,None,None,None,...,None,-9999.0,0.0,None,277.0,1.0,0.003610108303249,337.0,0.0,0.0
4,1366776_78053_16-23_2023-11-02 22:21:34.799,1366776,78053,2023-11-02 22:21:34.799,2023-11-02,0,1.0,None,None,None,...,None,-9999.0,0.0,None,359.0,0.0,0.0,337.0,0.0,0.0


In [5]:
numeric_data = numeric_data.dropna(how='all', axis=1)

In [6]:
# 3. Handle categorical features by encoding them to numeric
categorical_columns = numeric_data.select_dtypes(include=['object']).columns
numeric_data[categorical_columns] = numeric_data[categorical_columns].apply(lambda col: pd.Categorical(col).codes)

C:\Users\rishm\AppData\Local\Temp\ipykernel_10852\2417738053.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  numeric_data[categorical_columns] = numeric_data[categorical_columns].apply(lambda col: pd.Categorical(col).codes)


In [7]:
numeric_data

,id1,id2,id3,id4,id5,y,f1,f2,f3,f4,...,f357,f358,f359,f360,f361,f362,f363,f364,f365,f366
0,294845,19208,62,537713,1,0,0,-1,-1,-1,...,-1,0,0,-1,1048,0,0,2303,0,0
1,294876,19208,651,298195,0,0,0,-1,-1,-1,...,-1,-1,0,-1,1805,0,0,15,112,1481
2,294851,19208,193,1094,0,0,0,-1,-1,-1,...,-1,-1,0,-1,886,0,0,15,112,1481
3,294861,19208,377,537646,1,0,0,-1,-1,-1,...,-1,0,0,-1,1041,1,1058,2303,0,0
4,294870,19208,526,537649,1,0,0,-1,-1,-1,...,-1,0,0,-1,1194,0,0,2303,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
770159,757876,45788,637,361631,1,0,-1,-1,-1,-1,...,1397,391,0,-1,1839,1,2509,1930,1,2694
770160,757873,45788,279,361642,1,0,-1,-1,-1,-1,...,45,918,0,-1,1149,0,0,3203,1,8180
770161,757872,45788,109,361656,1,0,-1,-1,-1,-1,...,715,879,0,-1,1149,0,0,3203,1,8180
770162,761773,45977,703,327896,1,0,-1,-1,-1,-1,...,-1,-1,0,-1,-1,-1,-1,-1,-1,-1


In [8]:
imputer = KNNImputer(n_neighbors=5)
imputed_numeric_data = pd.DataFrame(imputer.fit_transform(numeric_data), columns=numeric_data.columns)

In [36]:
numerical_features = numeric_data.columns

In [38]:
# 5. Feature Engineering
poly = PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)
poly_features = poly.fit_transform(imputed_numeric_data[numerical_features])
poly_columns = poly.get_feature_names_out(numerical_features)
poly_df = pd.DataFrame(poly_features, columns=poly_columns)

In [40]:
y = to_categorical(numeric_data['y'], num_classes=2)

In [41]:
# 6. Feature Scaling: Standardize the numerical features
scaler = StandardScaler()
numeric_data[numerical_features] = scaler.fit_transform(numeric_data[numerical_features])

C:\Users\rishm\AppData\Local\Temp\ipykernel_10852\3017048166.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  numeric_data[numerical_features] = scaler.fit_transform(numeric_data[numerical_features])


In [42]:
# 7. Define features (X) and target (y)
X = numeric_data[numerical_features]

In [43]:
# Sequence Generation: Create time-series sequences (assuming fixed length of 10)
def create_sequences(X, y, time_steps=10):
    Xs, ys = [], []
    for i in range(len(X) - time_steps):
        Xs.append(X.iloc[i:i + time_steps].values)
        ys.append(y[i + time_steps])
    return np.array(Xs), np.array(ys)

# Create sequences for training, validation, and test sets
X_seq, y_seq = create_sequences(X, y)

In [44]:
# SMOTE for Class Imbalance
smote = SMOTE(random_state=42)
X_flat = X_seq.reshape(X_seq.shape[0], -1)  # Flatten data for SMOTE
y_flat = y_seq.argmax(axis=1)  # Convert one-hot encoding to class labels

In [45]:
# Apply SMOTE to balance classes
X_smote, y_smote = smote.fit_resample(X_flat, y_flat)

MemoryError: Unable to allocate 2.18 GiB for an array with shape (696052, 420) and data type float64

In [ ]:
# Reshape X back to its original time-series 3D shape
X_smote = X_smote.reshape(-1, X_seq.shape[1], X_seq.shape[2])

In [ ]:
# Convert y_smote back to one-hot encoding
y_smote = to_categorical(y_smote, num_classes=2)

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(X_smote, y_smote, test_size=0.3, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

In [ ]:
# Calculate class weights
class_weights = compute_class_weight('balanced', classes=np.unique(y_smote.argmax(axis=1)), y=y_smote.argmax(axis=1))
class_weights_dict = dict(enumerate(class_weights))

In [ ]:
# 8. Model Architecture (CNN + GRU + LSTM with Attention)
input_layer = Input(shape=(X_train.shape[1], X_train.shape[2]))  # Automatically adjust based on X_train's actual shape

In [ ]:
# CNN Layers with Batch Normalization
conv_layer = Conv1D(filters=64, kernel_size=2, activation='relu')(input_layer)
conv_layer = BatchNormalization()(conv_layer)
pool_layer = MaxPooling1D(pool_size=2)(conv_layer)

In [ ]:
# GRU Layer with Dropout
gru_layer = GRU(32, return_sequences=True)(pool_layer)
gru_layer = Dropout(0.4)(gru_layer)
gru_layer = BatchNormalization()(gru_layer)

In [ ]:
# Bidirectional LSTM Layer with Dropout
bi_lstm_layer = Bidirectional(LSTM(32, return_sequences=True))(gru_layer)
bi_lstm_layer = Dropout(0.4)(bi_lstm_layer)

In [ ]:
# Attention Layer
attention_output = Attention()([bi_lstm_layer, bi_lstm_layer])

In [ ]:
# Flatten the output
flattened_output = Flatten()(attention_output)

In [ ]:
# Dense Layers with L2 Regularization and Dropout
dense_layer_1 = Dense(64, activation='relu', kernel_regularizer='l2')(flattened_output)
dropout_layer_1 = Dropout(0.6)(dense_layer_1)
output_layer = Dense(2, activation='softmax')(dropout_layer_1)  # Output layer for 5 classes

In [ ]:
# Compile Model
optimizer = Adam(learning_rate=0.0001, clipnorm=1.0)
model = Model(inputs=input_layer, outputs=output_layer)
model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])

In [ ]:
# Train the model with class weights
history = model.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=140, batch_size=64, class_weight=class_weights_dict)

In [ ]:
import xgboost as xgb
xgb_clf = xgb.XGBClassifier(
    n_estimators=100,   # Number of trees
    learning_rate=0.1,  # Step size shrinkage
    max_depth=6,        # Maximum depth of trees
    subsample=0.8,      # Fraction of samples used per tree
    colsample_bytree=0.8, # Fraction of features used per tree
    random_state=42,
    use_label_encoder=False,
    eval_metric="logloss"  # Suppresses warning for older versions
)
xgb_clf.fit(X_train.reshape(X_train.shape[0], -1), y_train.argmax(axis=1))

In [ ]:
# 9. Model Evaluation (Accuracy, F1-score, MCC, Precision-Recall, AUC-PR)
y_pred = xgb_clf.predict(X_test.reshape(X_test.shape[0], -1))
y_score = xgb_clf.predict_proba(X_test.reshape(X_test.shape[0], -1))

In [ ]:
# F1-Score, MCC, and Accuracy
f1 = f1_score(y_test.argmax(axis=1), y_pred, average='weighted')
mcc = matthews_corrcoef(y_test.argmax(axis=1), y_pred)
accuracy = accuracy_score(y_test.argmax(axis=1), y_pred)
print(f'Accuracy: {accuracy:.4f}, F1-score: {f1:.4f}, MCC: {mcc:.4f}')

In [ ]:
data_1 = pd.read_parquet('test_data.parquet')

In [ ]:
Z_t=data_1.copy()

In [ ]:
Z_t

In [ ]:
# 3. Handle categorical features by encoding them to numeric
categorical_columns = data_1.select_dtypes(include=['object']).columns
data_1[categorical_columns] = data_1[categorical_columns].apply(lambda col: pd.Categorical(col).codes)

In [ ]:
numerical_features_1 = data_1.columns

In [ ]:
# 6. Feature Scaling: Standardize the numerical features
scaler = StandardScaler()
data_1[numerical_features_1] = scaler.fit_transform(data_1[numerical_features_1])

In [ ]:
Z_test = data_1[numerical_features_1]

In [ ]:
# Sequence Generation: Create time-series sequences (assuming fixed length of 10)
def create_sequences(Z_test, time_steps=10):
    Zs = []
    for i in range(len(Z_test) - time_steps):
        Zs.append(Z_test.iloc[i:i + time_steps].values)
    return np.array(Zs)

# Create sequences for training, validation, and test sets
Z_seq = create_sequences(Z_test)

In [ ]:
y_pred_1 = xgb_clf.predict(Z_seq.reshape(Z_seq.shape[0], -1))

In [ ]:
print(y_pred_1)

In [ ]:
Z_t2 = Z_t.iloc[10:].reset_index(drop=True)
results_df = pd.DataFrame({
    'id1': Z_t2['id1'],
    'id2': Z_t2['id2'],
    'id3': Z_t2['id3'],
    'id5': Z_t2['id5'],
    'pred': y_pred_1
})

In [ ]:
Z_t1=Z_t.iloc[:10].reset_index(drop=True)
results = pd.DataFrame({
    'id1': Z_t1['id1'],
    'id2': Z_t1['id2'],
    'id3': Z_t1['id3'],
    'id5': Z_t1['id5'],
    'pred': [1,1,1,1,1,1,1,1,1,1]
})

In [ ]:
results_df_1 = pd.concat([results, results_df], ignore_index=True)

In [ ]:
results_df_1

In [ ]:
results_df_1.to_excel("results_11.xlsx", index=False)